# Archiver utility example: DIAG0 cameras

This notebook shows how to get archived PV values for a device using
`slac_db.archiver_utility`. We use the profile monitor cameras in **DIAG0**.

Steps:
1. Find the camera devices in DIAG0.
2. List each camera's PVs (its accessors).
3. Get the current archived value of every PV.
4. Get values from an earlier time.

In [1]:
# Use the slac_db in this repo, not the shared AFS install
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd

import slac_db.device
from slac_db.archiver_utility import (
    get_archived_values_for_device,
    get_archived_values_for_pvs,
)

## 1. Find the cameras in DIAG0
Cameras (profile monitors) have the device type `PROF`.

In [3]:
diag0_cameras = slac_db.device.get_devices(area="DIAG0", device_type="PROF")
diag0_cameras

['OTRDG02', 'OTRDG04']

## 2. List a camera's PVs
Each accessor is a short, readable name for one of the device's PVs.

In [4]:
camera_name = diag0_cameras[0]
slac_db.device.get_all_accessors(camera_name)

{'filter_1_control': 'OTRS:DIAG0:420:FLT1_CTRL',
 'filter_1_status': 'OTRS:DIAG0:420:FLT1_STS',
 'filter_2_control': 'OTRS:DIAG0:420:FLT2_CTRL',
 'filter_2_status': 'OTRS:DIAG0:420:FLT2_STS',
 'image': 'OTRS:DIAG0:420:Image:ArrayData',
 'lamp_power': 'OTRS:DIAG0:420:TGT_LAMP_PWR',
 'n_bits': 'OTRS:DIAG0:420:N_OF_BITS',
 'n_col': 'OTRS:DIAG0:420:Image:ArraySize1_RBV',
 'n_row': 'OTRS:DIAG0:420:Image:ArraySize0_RBV',
 'orient_x': 'OTRS:DIAG0:420:X_ORIENT',
 'orient_y': 'OTRS:DIAG0:420:Y_ORIENT',
 'ref_rate': 'OTRS:DIAG0:420:ArrayRate_RBV',
 'ref_rate_vme': 'OTRS:DIAG0:420:FRAME_RATE',
 'resolution': 'OTRS:DIAG0:420:RESOLUTION',
 'sys_type': 'OTRS:DIAG0:420:SYS_TYPE',
 'target_control': 'OTRS:DIAG0:420:PNEUMATIC',
 'target_status': 'OTRS:DIAG0:420:TGT_STS'}

## 3. Get the current archived value of every PV on a camera
`get_archived_values_for_device` does the accessor lookup and the archiver request.
A value of `None` means the archiver has no data for that PV. For example,
the `image` waveform is usually not archived.

In [ ]:
camera_values = get_archived_values_for_device(camera_name)
pd.DataFrame.from_dict(camera_values, orient="index")

ArchiveRetrievalError: 2 PV(s) not found in archiver response: ['OTRS:DIAG0:420:Image:ArrayData', 'OTRS:DIAG0:420:SYS_TYPE']

### All DIAG0 cameras side by side
Each column is one camera. Each row is one accessor.

In [ ]:
values_by_camera = {
    camera: {
        accessor: reading["value"]
        for accessor, reading in get_archived_values_for_device(camera).items()
    }
    for camera in diag0_cameras
}
pd.DataFrame(values_by_camera)

## 4. Values from an earlier time
Pass `at_time` as a UTC timestamp like `"2026-10-10 12:00:00 UTC"` to get the value each PV had at that time.

In [ ]:
values_last_month = get_archived_values_for_device(
    camera_name, at_time="2026-10-10 12:00:00 UTC"
)
pd.DataFrame.from_dict(values_last_month, orient="index")

## Bonus: query PVs directly
If you already have PV names, use `get_archived_values_for_pvs`.

In [ ]:
get_archived_values_for_pvs([
    "OTRS:DIAG0:420:TGT_STS",
    "OTRS:DIAG0:525:TGT_STS",
])

In [ ]:
get_archived_values_for_device('QE01')

In [ ]:
get_archived_values_for_device('OTR2')

In [ ]:
get_archived_values_for_device('OTRDG02')

In [ ]:
get_archived_values_for_device('BX31')

In [ ]:
# for magnets get BDES unit, append EGU and query archiver for EGU's
# no bmax/bmin
# image array sizes are using the zoom cropped values from archiver, so we cant get nominal screen sizes from archiver
# can get resolution

In [ ]:
from slac_tools import get_archived

In [ ]:
get_archived('BEND:LTUH:220:BACT')